# German Bundestag: complete MP list

Extract one record for every person in the official `MdB-Stammdaten.xml` file, including former members from earlier election periods. The source archive and generated CSV are in `../datasets/`.

In [ ]:
from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET

import pandas as pd

DATA_DIR = Path("../datasets")
XML_ZIP = DATA_DIR / "MdB-Stammdaten.zip"
OUTPUT_CSV = DATA_DIR / "Bundestag_MPs_all.csv"

with zipfile.ZipFile(XML_ZIP) as archive:
    xml_name = next(name for name in archive.namelist() if name.lower().endswith(".xml"))
    with archive.open(xml_name) as xml_file:
        root = ET.parse(xml_file).getroot()

print(f"Loaded {xml_name}")

## Extract every member

The XML contains one `MDB` element per person. Keep every record, select the latest available name, retain the biographical fields, and list all associated election periods.

In [ ]:
def flatten_fields(element, prefix=""):
    """Return the direct text fields of an XML element as a dictionary."""
    if element is None:
        return {}
    return {
        prefix + child.tag: (child.text or "").strip() or None
        for child in element
    }


rows = []
for member in root.findall("MDB"):
    names = member.findall("NAMEN/NAME")
    latest_name = next(
        (name for name in names if not (name.findtext("HISTORIE_BIS") or "").strip()),
        names[-1] if names else None,
    )
    biography = member.find("BIOGRAFISCHE_ANGABEN")
    periods = [
        (period.findtext("WP") or "").strip()
        for period in member.findall("WAHLPERIODEN/WAHLPERIODE")
    ]

    row = {"ID": member.findtext("ID")}
    row.update(flatten_fields(latest_name, "NAME_"))
    row.update(flatten_fields(biography))
    row["WAHLPERIODEN"] = ", ".join(period for period in periods if period)
    row["ANZAHL_WAHLPERIODEN"] = len(periods)
    rows.append(row)

all_mps = pd.DataFrame(rows).dropna(axis=1, how="all")
if all_mps["ID"].isna().any() or all_mps["ID"].duplicated().any():
    raise ValueError("The XML must contain one unique ID for every MP record.")

all_mps.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Extracted {len(all_mps):,} MPs to {OUTPUT_CSV}")
all_mps.head()